# Data Preprocessing

Dataset: **Bank Marketing**

This notebook corresponds to **Section 3.4 "Data Preprocessing"** of the paper.
It transforms the cleaned dataset (`bank-cleaned.csv`) into the form consumed
by the non-private and Central DP modeling notebooks (`bank-processed.csv`):

1. **Month conversion** — `'jan'..'dec'` to integers `1..12`.
2. **One-hot encoding** — remaining categorical variables (`job`, `marital`,
   `education`, `contact`, `poutcome`) using `drop_first=True`.
3. **MinMax scaling** — numerical features rescaled to `[0, 1]`.

The output `bank-processed.csv` is the input to:
- `04_non_private_training.ipynb` (Section 4)
- `06_central_dp_training.ipynb` (Section 6)

> The Local DP notebook (`05_local_dp_training.ipynb`) does **not** consume
> this artifact: LDP requires the non-one-hot categorical form, so it loads
> `bank-cleaned.csv` directly and applies its own discretization step
> (Section 5.1).


In [1]:
from pathlib import Path
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# Paths
ROOT = Path.cwd().resolve()
DATA_PROCESSED = ROOT.parent / "data" / "processed"

# Numerical columns used downstream
NUMERIC_COLUMNS = ["age", "balance", "day", "duration", "campaign", "pdays", "previous"]



### Load cleaned dataset


In [2]:
data = pd.read_csv(DATA_PROCESSED / "bank-cleaned.csv")
print("Shape:", data.shape)


Shape: (45211, 17)


### Month conversion


In [3]:
# Month conversion (string -> integer)
data['month'] = data['month'].replace({
    'jan': 1, 'feb': 2, 'mar': 3, 'apr': 4,
    'may': 5, 'jun': 6, 'jul': 7, 'aug': 8,
    'sep': 9, 'oct': 10, 'nov': 11, 'dec': 12
}).infer_objects()


C:\Users\danie\AppData\Local\Temp\ipykernel_9500\3931924680.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data['month'] = data['month'].replace({


### One-hot encoding of remaining categorical variables


In [4]:
# One-hot encoding of remaining categorical columns
data_obj = data.select_dtypes(include=['object']).astype('category')
data = pd.get_dummies(data=data, columns=data_obj.columns, drop_first=True, dtype=int)


In [5]:
print('Number of features after one-hot encoding: %d' % (data.shape[1]))
data.head()


Number of features after one-hot encoding: 33


,age,default,balance,housing,loan,day,month,duration,campaign,pdays,...,marital_married,marital_single,education_secondary,education_tertiary,education_unknown,contact_telephone,contact_unknown,poutcome_other,poutcome_success,poutcome_unknown
0,58,0,2143,1,0,5,5,261,1,-1,...,1,0,0,1,0,0,1,0,0,1
1,44,0,29,1,0,5,5,151,1,-1,...,0,1,1,0,0,0,1,0,0,1
2,33,0,2,1,1,5,5,76,1,-1,...,1,0,1,0,0,0,1,0,0,1
3,47,0,1506,1,0,5,5,92,1,-1,...,1,0,0,0,1,0,1,0,0,1
4,33,0,1,0,0,5,5,198,1,-1,...,0,1,0,0,1,0,1,0,0,1


### MinMax scaling

Numerical features are rescaled to the range `[0, 1]` using `MinMaxScaler`.
The scaler is fit on the full dataset, matching the original implementation
in `04_non_private_training.ipynb` and `06_central_dp_training.ipynb`.


In [6]:
# MinMax scaling on numerical features
data[NUMERIC_COLUMNS] = MinMaxScaler().fit_transform(data[NUMERIC_COLUMNS])


### Save processed dataset


In [7]:
processed_path = DATA_PROCESSED / "bank-processed.csv"
data.to_csv(processed_path, index=False)
print(f"Processed dataset saved to: {processed_path}")


Processed dataset saved to: C:\Users\danie\OneDrive\Documentos\1 UNIANDES\PAPER DP\differential-privacy-data-analysis\BankMarketing\data\processed\bank-processed.csv
